<a href="https://colab.research.google.com/github/carloshsieh22/musicbehindthemedal/blob/main/DATASCI_112_Final_Project_Carlos_Allison_Data_Cleaning_and_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Collection from International Skating Union (ISU) Website

We pulled from 3 different ISU websites, each containing different features of interest.


*   Name, Gender, Country, Rank
* PCS (Program Component Score), PR (Performance), CO (Composition), Technical Element Score (TES)
* Categorized songs into 3 categories: Pop/Contemporary, Warhorse/Classical, Movie/Musical




In [ ]:
!pip install beautifulsoup4 requests pandas -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

def create_name_variants(name):
    name = ' '.join(name.split()).strip()
    variants = [name]  # Original

    parts = name.split()
    if len(parts) >= 2:
        last_first = f"{parts[-1].upper()} {' '.join(parts[:-1])}"
        variants.append(last_first)

        first_last = f"{' '.join(parts[:-1])} {parts[-1].upper()}"
        variants.append(first_last)

        if parts[0].isupper() and len(parts[0]) > 1:
            alt = f"{' '.join(parts[1:])} {parts[0]}"
            variants.append(alt)

    return list(set(variants))

def find_score_by_name(name, scores_dict):
    if name in scores_dict:
        return scores_dict[name]
    variants = create_name_variants(name)
    for variant in variants:
        if variant in scores_dict:
            return scores_dict[variant]
    return 'N/A'

def fetch_html_from_url(url, timeout=30):
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response.content.decode('utf-8')
    except Exception as e:
        print(f"Error fetching URL: {str(e)}")  # Enhanced error message
        return None

def scrape_segment_scores(html_content, segment_type):
    try:
        soup = BeautifulSoup(html_content, 'html.parser')
        scores = {}

        table = soup.find('table', class_='res_table') or soup.find('table', class_='MainTab')
        if not table:
            print(f"Warning: Could not find table for {segment_type}")
            return scores

        rows = table.find_all('tr')
        for row in rows[1:]:  # Skip header row
            cells = row.find_all('td')
            if len(cells) < 5:
                continue

            # Find the name cell (contains link with /bios/)
            name_link = None
            name_cell_idx = None
            for idx, cell in enumerate(cells):
                link = cell.find('a', href=lambda x: x and '/bios/' in x)
                if link:
                    name_link = link
                    name_cell_idx = idx
                    break

            if not name_link or name_cell_idx is None:
                continue

            skater_name = ' '.join(name_link.get_text(strip=True).split())
            rank = cells[0].get_text(strip=True)  # Rank is always first cell

            # Determine if there's a Q column (check if cell before name is 'Q')
            has_qualifier = (name_cell_idx > 1 and
                           cells[name_cell_idx - 1].get_text(strip=True) == 'Q')

            if has_qualifier:
                total_idx = name_cell_idx + 2
                tes_idx = name_cell_idx + 3
                pcs_idx = name_cell_idx + 5  # Skip empty cell
                co_idx = name_cell_idx + 6
                pr_idx = name_cell_idx + 7
            else:
                total_idx = name_cell_idx + 2
                tes_idx = name_cell_idx + 3
                pcs_idx = name_cell_idx + 5  # Skip empty cell
                co_idx = name_cell_idx + 6
                pr_idx = name_cell_idx + 7

            # Extract scores
            score = None
            tes_score = None
            pcs_score = None

            # Get total score
            if total_idx < len(cells):
                total_text = cells[total_idx].get_text(strip=True)
                if re.match(r'^\d+\.\d+$', total_text):
                    try:
                        score_val = float(total_text)
                        if 20 <= score_val <= 250:
                            score = score_val
                    except ValueError:
                        pass

            # Get TES (Technical Elements Score)
            if tes_idx < len(cells):
                tes_text = cells[tes_idx].get_text(strip=True)
                if re.match(r'^\d+\.\d+$', tes_text):
                    try:
                        tes_score = float(tes_text)
                    except ValueError:
                        pass

            # Get PCS (Program Components Score)
            if pcs_idx < len(cells):
                pcs_text = cells[pcs_idx].get_text(strip=True)
                if re.match(r'^\d+\.\d+$', pcs_text):
                    try:
                        pcs_score = float(pcs_text)
                    except ValueError:
                        pass

            # Extract CO and PR scores
            co_score = cells[co_idx].get_text(strip=True) if co_idx < len(cells) else None
            pr_score = cells[pr_idx].get_text(strip=True) if pr_idx < len(cells) else None

            if skater_name and score is not None:
                scores[skater_name] = {
                    'score': score,
                    'rank': rank,
                    'TES': tes_score,
                    'PCS': pcs_score,
                    'CO': co_score,
                    'PR': pr_score,
                }

        return scores
    except Exception as e:
        print(f"Error parsing {segment_type} scores: {str(e)}")
        import traceback
        traceback.print_exc()
        return {}

def scrape_competition_results(html_content, competition_name):
    soup = BeautifulSoup(html_content, 'html.parser')
    main_table = soup.find('table', class_='MainTab')

    if not main_table:
        print("Warning: Could not find main results table")
        return []

    # Locate the second nested table containing results
    nested_table = main_table.find('table')

    if not nested_table:
        print("Warning: Could not find nested results table")
        return []

    rows = nested_table.find_all('tr')
    skaters_data = []

    for row in rows[1:]:  # Skip the header row
        cells = row.find_all('td')
        if len(cells) >= 6:
            link = cells[1].find('a', href=lambda x: x and '/bios/' in x)
            if link:
                skater_name = link.get_text(strip=True)
                nation = cells[2].get_text(strip=True)  # Get nation from the 3rd cell

                # Try to fetch total points, ensuring it's not empty
                total_points_text = cells[3].get_text(strip=True)  # Assuming this is points
                try:
                    total_points = float(total_points_text)
                except ValueError:
                    total_points = None  # Set to None or a default value if conversion fails

                skaters_data.append({
                    'Skater Name': skater_name,
                    'Bio URL': link['href'],
                    'Nation': nation,
                    'Total Points': total_points
                })

    return skaters_data

def scrape_individual_skater(url, skater_name):
    try:
        time.sleep(1)
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        sp_music_raw = "N/A"
        fs_music_raw = "N/A"

        for i, row in enumerate(soup.find_all('tr')):
            cells = row.find_all('td')
            if cells:
                text = cells[0].get_text(strip=True)
                if "Music Short Program" in text:
                    sp_music_raw = cells[1].get_text(strip=True) if len(cells) > 1 else sp_music_raw
                elif "Music Free Skating" in text or "Music Free Dance" in text:
                    fs_music_raw = cells[1].get_text(strip=True) if len(cells) > 1 else fs_music_raw

        return {
            'sp_music': sp_music_raw,
            'fs_music': fs_music_raw
        }

    except Exception as e:
        print(f"Error fetching music info for {skater_name}: {str(e)}")
        return {'sp_music': 'N/A', 'fs_music': 'N/A'}

def scrape_competition_from_urls(results_url, sp_url, fs_url, competition_name):
    print(f"Scraping: {competition_name}")
    results_html = fetch_html_from_url(results_url)
    if not results_html:
        return None

    sp_html = fetch_html_from_url(sp_url)
    if not sp_html:
        return None

    fs_html = fetch_html_from_url(fs_url)
    if not fs_html:
        return None

    # Scrape segment scores with TES, PCS, CO, and PR
    sp_scores = scrape_segment_scores(sp_html, "Short Program")
    fs_scores = scrape_segment_scores(fs_html, "Free Skating")
    skaters_data = scrape_competition_results(results_html, competition_name)

    all_rows = []
    for skater in skaters_data:
        skater_name = skater['Skater Name']
        sp_score_data = sp_scores.get(skater_name, {
            'score': 'N/A',
            'rank': 'N/A',
            'TES': 'N/A',
            'PCS': 'N/A',
            'CO': 'N/A',
            'PR': 'N/A'
        })
        fs_score_data = fs_scores.get(skater_name, {
            'score': 'N/A',
            'rank': 'N/A',
            'TES': 'N/A',
            'PCS': 'N/A',
            'CO': 'N/A',
            'PR': 'N/A'
        })

        all_rows.append({
            'Skater Name': skater_name,
            'Competition': competition_name,
            'Nation': skater['Nation'],
            'Total Points': skater['Total Points'],
            'SP Score': sp_score_data['score'],
            'SP Rank': sp_score_data['rank'],
            'SP TES': sp_score_data['TES'],
            'SP PCS': sp_score_data['PCS'],
            'SP CO': sp_score_data['CO'],
            'SP PR': sp_score_data['PR'],
            'FS Score': fs_score_data['score'],
            'FS Rank': fs_score_data['rank'],
            'FS TES': fs_score_data['TES'],
            'FS PCS': fs_score_data['PCS'],
            'FS CO': fs_score_data['CO'],
            'FS PR': fs_score_data['PR']
        })

    df = pd.DataFrame(all_rows)
    print(f"Total entries created: {len(df)}")
    return df
# BATCH SCRAPING: Multiple Competitions

def scrape_multiple_competitions(competitions_list):
    all_dfs = []

    for i, comp in enumerate(competitions_list, 1):
        print("\n" + "=" * 80)
        print(f"COMPETITION {i}/{len(competitions_list)}: {comp['name']}")
        print("=" * 80)

        df = scrape_competition_from_urls(
            results_url=comp['results_url'],
            sp_url=comp['sp_url'],
            fs_url=comp['fs_url'],
            competition_name=comp['name']
        )

        if df is not None and len(df) > 0:
            all_dfs.append(df)
            print(f"✓ Successfully scraped {comp['name']}")
        else:
            print(f"✗ Failed to scrape {comp['name']}")

        if i < len(competitions_list):
            print("\nPausing 5 seconds before the next competition...")
            time.sleep(5)

    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)

        return combined_df
    else:
        print("\n✗ No data scraped successfully")
        return None

# ALL COMPETITIONS TO SCRAPE - 2024-2025 Season - Women's Singles

competitions_to_scrape = [
    # ========================================================================
    # OLYMPICS 2026
    # ========================================================================
    {
        'name': 'Olympics 2026 - Women',
        'results_url': 'https://results.isu.org/results/season2526/owg2026/CAT002RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/owg2026/SEG003.htm',
        'fs_url': 'https://results.isu.org/results/season2526/owg2026/SEG004.htm'
    },
    {
        'name': 'Olympics 2026 - Men',
        'results_url': 'https://results.isu.org/results/season2526/owg2026/CAT001RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/owg2026/SEG001.htm',
        'fs_url': 'https://results.isu.org/results/season2526/owg2026/SEG002.htm'
    },

    # ========================================================================
    # GRAND PRIX SERIES 2025 (6 events)
    # ========================================================================
    {
        'name': 'GP Skate France 2025 - Women',
        'results_url': 'https://results.isu.org/results/season2526/gpfra2025/CAT002RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpfra2025/SEG003.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpfra2025/SEG001.htm'
    },
    {
        'name': 'GP Skate France 2025 - Men',
        'results_url': 'https://results.isu.org/results/season2526/gpfra2025/CAT001RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpfra2025/SEG001.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpfra2025/SEG002.htm'
    },
    {
        'name': 'GP Skate Canada 2025 - Women',
        'results_url': 'https://www.isuresults.com/results/season2526/gpcan2025/CAT002RS.htm',
        'sp_url': 'https://www.isuresults.com/results/season2526/gpcan2025/SEG003.htm',
        'fs_url': 'https://www.isuresults.com/results/season2526/gpcan2025/SEG004.htm'
    },
    {
        'name': 'GP Skate Canada 2025 - Men',
        'results_url': 'https://www.isuresults.com/results/season2526/gpcan2025/CAT001RS.htm',
        'sp_url': 'https://www.isuresults.com/results/season2526/gpcan2025/SEG001.htm',
        'fs_url': 'https://www.isuresults.com/results/season2526/gpcan2025/SEG002.htm'
    },
    {
        'name': 'GP Grand Prix USA 2025 - Women',
        'results_url': 'https://results.isu.org/results/season2526/gpusa2025/CAT002RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpusa2025/SEG003.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpusa2025/SEG004.htm'
    },
    {
        'name': 'GP Grand Prix USA 2025 - Men',
        'results_url': 'https://results.isu.org/results/season2526/gpusa2025/CAT001RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpusa2025/SEG001.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpusa2025/SEG002.htm'
    },
    {
        'name': 'GP NHK Trophy 2025 - Women',
        'results_url': 'https://www.isuresults.com/results/season2526/gpjpn2025/CAT002RS.htm',
        'sp_url': 'https://www.isuresults.com/results/season2526/gpjpn2025/SEG003.htm',
        'fs_url': 'https://www.isuresults.com/results/season2526/gpjpn2025/SEG004.htm'
    },
    {
        'name': 'GP NHK Trophy 2025 - Men',
        'results_url': 'https://www.isuresults.com/results/season2526/gpjpn2025/CAT001RS.htm',
        'sp_url': 'https://www.isuresults.com/results/season2526/gpjpn2025/SEG001.htm',
        'fs_url': 'https://www.isuresults.com/results/season2526/gpjpn2025/SEG002.htm'
    },
    {
        'name': 'GP Cup of China 2025 - Women',
        'results_url': 'https://results.isu.org/results/season2526/gpchn2025/CAT002RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpchn2025/SEG003.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpchn2025/SEG004.htm'
    },
    {
        'name': 'GP Cup of China 2025 - Men',
        'results_url': 'https://results.isu.org/results/season2526/gpchn2025/CAT001RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpchn2025/SEG001.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpchn2025/SEG002.htm'
    },
    {
        'name': 'GP Finlandia Trophy 2025 - Women',
        'results_url': 'https://results.isu.org/results/season2526/gpfin2025/CAT002RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpfin2025/SEG003.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpfin2025/SEG004.htm'
    },
    {
        'name': 'GP Finlandia Trophy 2025 - Men',
        'results_url': 'https://results.isu.org/results/season2526/gpfin2025/CAT001RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/gpfin2025/SEG001.htm',
        'fs_url': 'https://results.isu.org/results/season2526/gpfin2025/SEG002.htm'
    },

    # ========================================================================
    # FOUR CONTINENTS 2025
    # ========================================================================
    {
        'name': 'Four Continents 2025 - Women',
        'results_url': 'https://results.isu.org/results/season2526/fc2026/CAT002RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/fc2026/SEG003.htm',
        'fs_url': 'https://results.isu.org/results/season2526/fc2026/SEG004.htm'
    },
    {
        'name': 'Four Continents 2025 - Men',
        'results_url': 'https://results.isu.org/results/season2526/fc2026/CAT001RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/fc2026/SEG001.htm',
        'fs_url': 'https://results.isu.org/results/season2526/fc2026/SEG002.htm'
    },

    # ========================================================================
    # EUROPEAN CHAMPIONSHIPS 2025
    # ========================================================================
    {
        'name': 'European Championships 2025 - Women',
        'results_url': 'https://results.isu.org/results/season2526/ec2026/CAT002RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/ec2026/SEG003.htm',
        'fs_url': 'https://results.isu.org/results/season2526/ec2026/SEG004.htm'
    },
    {
        'name': 'European Championships 2025 - Men',
        'results_url': 'https://results.isu.org/results/season2526/ec2026/CAT001RS.htm',
        'sp_url': 'https://results.isu.org/results/season2526/ec2026/SEG001.htm',
        'fs_url': 'https://results.isu.org/results/season2526/ec2026/SEG002.htm'
    }
]

In [ ]:
print("FIGURE SKATING COMPETITION SCRAPER")

combined_df = scrape_multiple_competitions(competitions_to_scrape)

if combined_df is not None:
    output_file = 'all_competitions_2025_2026.csv'
    combined_df.to_csv(output_file, index=False, encoding='utf-8')
    print("Combined dataset saved.")
else:
    print("SCRAPING FAILED")
    print("No data was successfully scraped. Please check the URLs and try again.")

FIGURE SKATING COMPETITION SCRAPER

COMPETITION 1/18: Olympics 2026 - Women
Scraping: Olympics 2026 - Women
Total entries created: 29
✓ Successfully scraped Olympics 2026 - Women

Pausing 5 seconds before the next competition...

COMPETITION 2/18: Olympics 2026 - Men
Scraping: Olympics 2026 - Men
Total entries created: 29
✓ Successfully scraped Olympics 2026 - Men

Pausing 5 seconds before the next competition...

COMPETITION 3/18: GP Skate France 2025 - Women
Scraping: GP Skate France 2025 - Women
Total entries created: 12
✓ Successfully scraped GP Skate France 2025 - Women

Pausing 5 seconds before the next competition...

COMPETITION 4/18: GP Skate France 2025 - Men
Scraping: GP Skate France 2025 - Men
Total entries created: 12
✓ Successfully scraped GP Skate France 2025 - Men

Pausing 5 seconds before the next competition...

COMPETITION 5/18: GP Skate Canada 2025 - Women
Scraping: GP Skate Canada 2025 - Women
Total entries created: 12
✓ Successfully scraped GP Skate Canada 2025 - 

In [ ]:
combined_df[['Competition', 'Gender']] = combined_df['Competition'].str.split(' - ', n=1, expand=True)

combined_df

,Skater Name,Competition,Nation,Total Points,SP Score,SP Rank,SP TES,SP PCS,SP CO,SP PR,FS Score,FS Rank,FS TES,FS PCS,FS CO,FS PR,Gender
0,LIU Alysa,Olympics 2026,USA,None,76.59,3,41.34,35.25,8.75,9.11,150.2,1,77.74,72.46,8.96,9.32,Women
1,SAKAMOTO Kaori,Olympics 2026,JPN,None,77.23,2,40.08,37.15,9.25,9.29,147.67,2,72.83,74.84,9.21,9.43,Women
2,NAKAI Ami,Olympics 2026,JPN,None,78.71,1,45.02,33.69,8.25,8.54,140.45,9,72.53,67.92,8.29,8.54,Women
3,CHIBA Mone,Olympics 2026,JPN,None,74.00,4,38.72,35.28,8.89,8.89,143.88,4,74.46,69.42,8.54,8.64,Women
4,GLENN Amber,Olympics 2026,USA,None,67.39,13,34.19,33.20,8.39,8.07,147.52,3,78.87,68.65,8.43,8.64,Women
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,Jakub LOFEK,European Championships 2025,POL,None,59.76,25,27.97,31.79,6.36,6.39,N/A,N/A,N/A,N/A,N/A,N/A,Men
311,Alp Eren OZKAN,European Championships 2025,TUR,None,58.47,26,28.76,29.71,5.96,6.04,N/A,N/A,N/A,N/A,N/A,N/A,Men
312,Kevin AYMOZ,European Championships 2025,FRA,None,53.95,27,21.57,36.38,7.46,6.50,N/A,N/A,N/A,N/A,N/A,N/A,Men
313,Alexander ZLATKOV,European Championships 2025,BUL,None,50.90,28,21.14,29.76,6.00,5.89,N/A,N/A,N/A,N/A,N/A,N/A,Men


In [ ]:
combined_df.drop('Total Points', axis=1, inplace=True)
combined_df

,Skater Name,Competition,Nation,SP Score,SP Rank,SP TES,SP PCS,SP CO,SP PR,FS Score,FS Rank,FS TES,FS PCS,FS CO,FS PR,Gender
0,LIU Alysa,Olympics 2026,USA,76.59,3,41.34,35.25,8.75,9.11,150.2,1,77.74,72.46,8.96,9.32,Women
1,SAKAMOTO Kaori,Olympics 2026,JPN,77.23,2,40.08,37.15,9.25,9.29,147.67,2,72.83,74.84,9.21,9.43,Women
2,NAKAI Ami,Olympics 2026,JPN,78.71,1,45.02,33.69,8.25,8.54,140.45,9,72.53,67.92,8.29,8.54,Women
3,CHIBA Mone,Olympics 2026,JPN,74.00,4,38.72,35.28,8.89,8.89,143.88,4,74.46,69.42,8.54,8.64,Women
4,GLENN Amber,Olympics 2026,USA,67.39,13,34.19,33.20,8.39,8.07,147.52,3,78.87,68.65,8.43,8.64,Women
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,Jakub LOFEK,European Championships 2025,POL,59.76,25,27.97,31.79,6.36,6.39,N/A,N/A,N/A,N/A,N/A,N/A,Men
311,Alp Eren OZKAN,European Championships 2025,TUR,58.47,26,28.76,29.71,5.96,6.04,N/A,N/A,N/A,N/A,N/A,N/A,Men
312,Kevin AYMOZ,European Championships 2025,FRA,53.95,27,21.57,36.38,7.46,6.50,N/A,N/A,N/A,N/A,N/A,N/A,Men
313,Alexander ZLATKOV,European Championships 2025,BUL,50.90,28,21.14,29.76,6.00,5.89,N/A,N/A,N/A,N/A,N/A,N/A,Men


Uses a voting model (mode) for skaters with multiple song categories. For example if Skater X skates to 2 pop songs and a warhorse, Skater X skates to pop.

In [ ]:
def get_mode_category(df):
    return (
        df.groupby("Skater Name")[["Category", 'Song']]
        .agg(lambda x: x.mode().iloc[0])
        .reset_index()
        .rename(columns={"Category": "category"})
    )

sp_cat = get_mode_category(short_program_songs)
fs_cat = get_mode_category(free_skate_songs)

df = combined_df.copy()

df = df.merge(sp_cat.rename(columns={"category": "sp_category"}),
              on="Skater Name", how="left")
df = df.merge(fs_cat.rename(columns={"category": "fs_category"}),
              on="Skater Name", how="left")

In [ ]:
combined_df.to_csv('combined_df.csv')

In [ ]:
def scrape_music(html_content):
    """
    Extracts the music titles for Short Program and Free Skating from a given HTML content.
    Returns a dictionary with lists of song titles for both segments.
    """
    try:
        soup = BeautifulSoup(html_content, 'html.parser')
        music_info = {'sp_music': [], 'fs_music': []}

        # Locate all relevant rows
        rows = soup.find_all('tr')

        for row in rows:
            # Check for Short Program music
            if "Music Short Program" in row.text:
                next_row = row.find_next_sibling('tr')  # Get the next row for the list of songs
                if next_row:
                    # Get the text directly from the <td>
                    songs_cell = next_row.find('td')
                    if songs_cell:
                        # Use get_text() to get the content without HTML tags
                        songs_text = songs_cell.get_text(separator='<br>', strip=True)  # separator will prevent losing titles
                        music_info['sp_music'] = [song.strip() for song in songs_text.split('<br>') if song.strip()]

            # Check for Free Skating music
            if "Music Free Skating" in row.text or "Music Free Dance" in row.text:
                next_row = row.find_next_sibling('tr')  # Get the next row for the list of songs
                if next_row:
                    # Get the text directly from the <td>
                    songs_cell = next_row.find('td')
                    if songs_cell:
                        # Use get_text() to get the content without HTML tags
                        songs_text = songs_cell.get_text(separator='<br>', strip=True)  # separator prevents losing titles
                        music_info['fs_music'] = [song.strip() for song in songs_text.split('<br>') if song.strip()]

        return music_info

    except Exception as e:
        print(f"Error extracting music information: {str(e)}")
        return {
            'sp_music': ['N/A'],
            'fs_music': ['N/A']
        }

In [ ]:
def add_music_info_to_combined_df(df, competitions):
    """
    Fetches and adds music information for each skater in the given DataFrame
    using the competition result webpages.
    """
    music_data = []

    for index, row in df.iterrows():
        skater_name = row['Skater Name']
        competition_name = row['Competition']

        # Find the competition to get the relevant URLs
        competition = next((comp for comp in competitions if comp['name'] == competition_name), None)

        if competition:
            results_url = competition['results_url']
            results_html = fetch_html_from_url(results_url)
            if results_html:
                soup = BeautifulSoup(results_html, 'html.parser')
                skater_row = soup.find('a', text=re.compile(re.escape(skater_name), re.IGNORECASE)).find_parent('tr')

                if skater_row:
                    bio_link = skater_row.find('a', href=True)
                    if bio_link:
                        bio_url = "https://results.isu.org" + bio_link['href']
                        music_html = fetch_html_from_url(bio_url)
                        if music_html:
                            music_info = scrape_music(music_html)
                        else:
                            music_info = {'sp_music': 'N/A', 'fs_music': 'N/A'}
                    else:
                        music_info = {'sp_music': 'N/A', 'fs_music': 'N/A'}
                else:
                    print(f"Skater {skater_name} not found in results for {competition_name}.")
                    music_info = {'sp_music': 'N/A', 'fs_music': 'N/A'}
            else:
                print(f"Failed to fetch results for {competition_name}.")
                music_info = {'sp_music': 'N/A', 'fs_music': 'N/A'}
        else:
            print(f"Competition {competition_name} not found in the configured list.")
            music_info = {'sp_music': 'N/A', 'fs_music': 'N/A'}

        # Append the music information only if unique
        music_data.append({
            'Skater Name': skater_name,
            'SP Music': music_info['sp_music'],
            'FS Music': music_info['fs_music']
        })

        time.sleep(1)

    # Convert music_data to DataFrame
    music_df = pd.DataFrame(music_data)

    return music_df

In [ ]:
music_df = add_music_info_to_combined_df(combined_df, competitions_to_scrape)


/tmp/ipykernel_149/2846257818.py:20: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  skater_row = soup.find('a', text=re.compile(re.escape(skater_name), re.IGNORECASE)).find_parent('tr')


In [ ]:
music_df

,Skater Name,SP Music,FS Music
0,LIU Alysa,[Promise by Laufey],[MacArthur Park Suite by Donna Summer]
1,SAKAMOTO Kaori,[Time To Say Goodbye performed by Sarah Bright...,"[La vie en rose by Patricia Kaas, Édith Piaf, ..."
2,NAKAI Ami,[La Strada by Nino Rota],"[What a Wonderful World by Lexi Walker, The Pi..."
3,CHIBA Mone,"[Last Dance by Donna Summer, Giorgio Moroder, ...","[A Thousand Times Goodnight (from ""Romeo and J..."
4,GLENN Amber,[Like A Prayer by Madonna],"[I Will Find You by Audiomachine, The Return b..."
...,...,...,...
310,Jakub LOFEK,[Cold by Jonathan Roy feat. Kim Richardson],"[Pure Imagination (from ""Wonka"" soundtrack) by..."
311,Alp Eren OZKAN,[Red Right Hand by Nick Cave and the Bad Seeds...,"[The World Is Not Enough (from ""James Bond"" so..."
312,Kevin AYMOZ,"[Le Lac by Jean-Michel Blais, Judas by Lady Gaga]","[Boléro, m. 81 by Maurice Ravel]"
313,Alexander ZLATKOV,[Nureyev (from “The White Crow”) by Lisa Batia...,[Mala Luna by Gino Vannelli]


Reformats the music dataframe and merges it to skating results dataframe

In [ ]:
df_sp_exploded = music_df.explode('SP Music')

# Exploding FS Music
df_fs_exploded = music_df.explode('FS Music')

# Combine the exploded DataFrames into one, keeping track of the program type
df_sp_exploded = df_sp_exploded[['Skater Name', 'SP Music']]
df_fs_exploded = df_fs_exploded[['Skater Name', 'FS Music']]
df_combined = pd.concat([df_sp_exploded.assign(Type='Short Program'),
                          df_fs_exploded.assign(Type='Free Skate')], ignore_index=True)

# Reset index if desired
df_combined.reset_index(drop=True, inplace=True)

# Display the final DataFrame
df_combined

,Skater Name,SP Music,Type,FS Music
0,LIU Alysa,Promise by Laufey,Short Program,NaN
1,SAKAMOTO Kaori,Time To Say Goodbye performed by Sarah Brightm...,Short Program,NaN
2,NAKAI Ami,La Strada by Nino Rota,Short Program,NaN
3,CHIBA Mone,"Last Dance by Donna Summer, Giorgio Moroder, P...",Short Program,NaN
4,GLENN Amber,Like A Prayer by Madonna,Short Program,NaN
...,...,...,...,...
1156,Alp Eren OZKAN,NaN,Free Skate,"From Russia With Love (""James Bond"" soundtrack)"
1157,Kevin AYMOZ,NaN,Free Skate,"Boléro, m. 81 by Maurice Ravel"
1158,Alexander ZLATKOV,NaN,Free Skate,Mala Luna by Gino Vannelli
1159,David SEDEJ,NaN,Free Skate,The Great Adventure by Rok Nardin
